In [13]:
import pandas as pd
df = pd.read_csv("../data/raw/results.csv")
df.info()
df.head()


<class 'pandas.DataFrame'>
RangeIndex: 49940 entries, 0 to 49939
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        49940 non-null  str  
 1   home_team   49940 non-null  str  
 2   away_team   49940 non-null  str  
 3   home_score  49940 non-null  int64
 4   away_score  49940 non-null  int64
 5   tournament  49940 non-null  str  
 6   city        49940 non-null  str  
 7   country     49940 non-null  str  
 8   neutral     49940 non-null  bool 
dtypes: bool(1), int64(2), str(6)
memory usage: 5.9 MB


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


In [15]:
df.isna().sum()

date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64

In [18]:
print("Doublons :", df.duplicated().sum())
print("Période :", df["date"].min(), "->", df["date"].max())
print("Nombre d'équipes :", pd.concat([df["home_team"], df["away_team"]]).nunique())
former = pd.read_csv("../data/raw/former_names.csv")
former

Doublons : 0
Période : 1872-11-30 -> 2026-10-06
Nombre d'équipes : 337


,current,former,start_date,end_date
0,Benin,Dahomey,1959-11-08,1975-11-30
1,Burkina Faso,Upper Volta,1960-04-14,1984-08-04
2,Curaçao,Netherlands Antilles,1957-03-03,2010-10-10
3,Czechoslovakia,Bohemia,1903-04-05,1919-01-01
4,Czechoslovakia,Bohemia and Moravia,1939-01-01,1945-05-01
5,Czechoslovakia,Representation of Czechs and Slovaks,1993-03-24,1993-11-17
6,DR Congo,Belgian Congo,1948-05-25,1956-01-02
7,DR Congo,Congo-Léopoldville,1963-04-12,1964-07-19
8,DR Congo,Congo-Kinshasa,1965-01-09,1970-11-24
9,DR Congo,Zaïre,1971-01-10,1997-04-27


In [19]:
team_df = pd.read_csv("../data/processed/team_matches.csv", parse_dates=["date"])
team_df.info()
print("Taux de victoire global :", team_df["win"].mean())
print(team_df.groupby("is_home")["win"].mean())
print(team_df.groupby("neutral")["win"].mean())
team_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 65642 entries, 0 to 65641
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   match_id       65642 non-null  int64         
 1   date           65642 non-null  datetime64[us]
 2   team           65642 non-null  str           
 3   opponent       65642 non-null  str           
 4   goals_for      65642 non-null  int64         
 5   goals_against  65642 non-null  int64         
 6   is_home        65642 non-null  int64         
 7   neutral        65642 non-null  int64         
 8   tournament     65642 non-null  str           
 9   win            65642 non-null  int64         
dtypes: datetime64[us](1), int64(6), str(3)
memory usage: 7.2 MB
Taux de victoire global : 0.3827427561622132
is_home
0    0.313324
1    0.507278
Name: win, dtype: float64
neutral
0    0.382906
1    0.382331
Name: win, dtype: float64


,match_id,date,team,opponent,goals_for,goals_against,is_home,neutral,tournament,win
0,0,1990-01-12,Algeria,Mali,5,0,0,1,Friendly,1
1,0,1990-01-12,Mali,Algeria,0,5,0,1,Friendly,0
2,1,1990-01-14,Algeria,Cameroon,3,1,0,1,Friendly,1
3,1,1990-01-14,Cameroon,Algeria,1,3,0,1,Friendly,0
4,2,1990-01-17,Greece,Belgium,2,0,1,0,Friendly,1


## Taux de victoire selon le lieu du match

Le groupement par `neutral` ne montre rien : chaque match donne une ligne par équipe (une gagne, l'autre perd), donc la moyenne vaut toujours `(1 - part de nuls) / 2`. On crée donc une colonne `venue` à 3 valeurs (domicile / extérieur / neutre) pour mesurer le vrai effet du terrain.

In [ ]:
import numpy as np

team_df["venue"] = np.select(
    [team_df["neutral"] == 1, team_df["is_home"] == 1],
    ["neutre", "domicile"],
    default="exterieur",
)

venue_stats = team_df.groupby("venue")["win"].agg(["mean", "count"])
print(venue_stats)

# Part de nuls : 1 - 2 * taux de victoire global
print("Part de nuls :", round(1 - 2 * team_df["win"].mean(), 3))

# Référence naive : toujours prédire "ne gagne pas"
print("Accuracy du modèle naif 'jamais de victoire' :", round(1 - team_df["win"].mean(), 3))

**Commentaire :**
- Une équipe qui joue à domicile gagne environ 1 match sur 2, contre environ 1 sur 4 à l'extérieur : `is_home` sera une variable utile pour le modèle.
- Sur terrain neutre, le taux est symétrique (proche du taux global de 38 %), car aucune équipe n'a l'avantage du terrain.
- Environ 23 % des matchs sont nuls et comptent comme non-victoire : un modèle qui répond toujours « ne gagne pas » a déjà ~62 % d'accuracy, c'est la référence à battre.